# 11 · 트랙 A — 변이 유형 집계 피처

- **Owner**: member_d (iljun)
- **Experiment ID**: `member-d-logreg-002` ~ `-006` (ablation)
- **Track**: A · 변이 표기 활용
- **Seed**: 42
- **Validation**: StratifiedKFold-5
- **기준선**: `member-d-logreg-001` = **Macro F1 0.36305** / Accuracy 0.36026

## 가설

지금 베이스라인은 `WT / 변이` 이진화라 **어떤 유형의 변이인지**를 전부 버린다.
그런데 클래스별 변이 유형 구성이 실제로 다르다.

| 클래스 | silent 비율 | frameshift 비율 |
|---|---|---|
| DLBC | **41.20 %** | 1.94 % |
| *(전체)* | *26.25 %* | *3.89 %* |
| LAML | **17.68 %** | **15.24 %** |

silent 는 2.33배, frameshift 는 **20배** 벌어진다.
→ 유형별 카운트를 피처로 넣으면 점수가 오르는가?

## 이 노트북이 하는 일

1. 변이 문자열 파서 (팀 표준안으로 통일)
2. 피처 블록 4종을 만드는 `build_features()` — fold 안전
3. Leakage 자가검증
4. **Ablation** — 블록을 하나씩 켜가며 기여도 측정
5. 최고 조합에 모델 3종 비교
6. 최종 학습 · submission · metrics.json

> ⏱ 전체 Run All 기준 **3\~5분**. CV 8회(=40 fit)가 대부분.


## Section 1 · 설정

In [1]:
import sys, platform, time, json, re, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import sklearn
from scipy import sparse
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs" / "baseline.yaml").exists():
            return path
    raise FileNotFoundError("저장소 루트를 찾지 못했습니다. JupyterLab을 레포 루트에서 실행하세요.")


NB = "11"                                   # artifacts 파일명 접두
MEMBER = "member_d"
SEED = 42
TARGET, ID = "SUBCLASS", "ID"

ROOT = find_project_root(Path.cwd())
DATA = ROOT / "data" / "raw"
ARTIFACTS = Path.cwd() / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
submission = pd.read_csv(DATA / "sample_submission.csv")
gene_cols = [c for c in train.columns if c not in (TARGET, ID)]
y_all = train[TARGET].values

print(f"python {platform.python_version()} · sklearn {sklearn.__version__}")
print(f"train {train.shape} · test {test.shape} · 유전자 {len(gene_cols)}")

python 3.12.13 · sklearn 1.9.0
train (6201, 4386) · test (2546, 4385) · 유전자 4384


## Section 2 · 변이 문자열 파서

<callout>

### 팀 표준안으로 통일

같은 데이터를 두 사람이 다르게 파싱하면 트랙 병합에서 피처가 안 맞는다. 아래 두 가지를 규칙으로 고정한다.

**① 한 칸 안의 중복 토큰은 1개로 센다.** `"R248Q R248Q"` → 1건.
전사체가 여러 개라 같은 변이가 중복 표기됐을 가능성이 높고, 중복을 세면 카운트가 부풀려진다. (총 6,100건 해당)

**② 판정 순서를 고정한다.** `>` → `fs` → `del`/`ins` → 나머지.
`^([A-Z*]+)(\d+)(.*)$` 로 파싱하고 `*261*`(종결→종결)은 `other` 로 뺀다.
앞 아미노산을 1글자로만 받으면 `TP469fs` 같은 두 글자 접두가 통째로 빠진다.

</callout>

In [2]:
PATTERN = re.compile(r"^([A-Z*]+)(\d+)(.*)$")
KINDS = ["missense", "silent", "nonsense", "frameshift", "indel", "other"]


def classify(token: str) -> str:
    """변이 토큰 하나를 기능적 유형으로 분류한다. 판정 순서가 중요하다."""
    if ">" in token:
        return "other"                       # 468_469LG>F* 복합 치환
    if "fs" in token:
        return "frameshift"
    if "del" in token or "ins" in token:
        return "indel"
    m = PATTERN.match(token)
    if not m:
        return "other"
    ref, _, alt = m.groups()
    if ref == "*" and alt == "*":
        return "other"                       # *261* 종결→종결
    if alt == "*":
        return "nonsense"
    if alt == ref:
        return "silent"
    if alt == "":
        return "other"
    return "missense"


def parse_sample_counts(df: pd.DataFrame) -> pd.DataFrame:
    """샘플별 유형 카운트 + 부담 지표. 전부 '한 행 안에서' 끝나므로 Leakage 가 아니다."""
    raw = df[gene_cols].fillna("WT").values
    mask = raw != "WT"
    n = len(df)
    out = np.zeros((n, len(KINDS) + 3), dtype=np.float32)
    for i in range(n):
        cells_i = raw[i][mask[i]]
        c = Counter()
        n_events = 0
        n_multi = 0
        for s in cells_i:
            toks = set(s.split())            # ← 규칙 ① 칸 내부 중복 제거
            if len(toks) > 1:
                n_multi += 1
            n_events += len(toks)
            for t in toks:
                c[classify(t)] += 1
        for j, k in enumerate(KINDS):
            out[i, j] = c[k]
        out[i, len(KINDS) + 0] = mask[i].sum()   # 변이 유전자 수
        out[i, len(KINDS) + 1] = n_events        # 총 이벤트 수
        out[i, len(KINDS) + 2] = n_multi         # 다중 이벤트 유전자 수
    cols = KINDS + ["n_mut_genes", "n_events", "n_multi_genes"]
    return pd.DataFrame(out, columns=cols, index=df.index)


t0 = time.time()
cnt_train = parse_sample_counts(train)
cnt_test = parse_sample_counts(test)
print(f"파싱 완료 {time.time() - t0:.0f}s")
print()
print("train 유형별 총계 (팀 표준안)")
print(cnt_train[KINDS].sum().astype(int).to_string())
print(f"\n합계 {int(cnt_train[KINDS].sum().sum()):,}")

파싱 완료 3s

train 유형별 총계 (팀 표준안)
missense      161051
silent         64844
nonsense       13080
frameshift      9764
indel              3
other            322

합계 249,064


## Section 3 · 피처 블록

네 블록을 조합해 ablation 한다. **모두 행 내부 연산**이라 Leakage 가 아니다.

| 블록 | 내용 | 차원 |
|---|---|---|
| **G** | 유전자 이진화 (train fold 에서 상수열 제거) | ~4,230 |
| **B** | 변이 부담 — `log1p` 유전자수 / 이벤트수 / 다중유전자수 | 3 |
| **V** | 유형별 카운트 `log1p` | 6 |
| **R** | 유형별 **비율** (총 이벤트 대비) | 6 |

**R 을 따로 두는 이유** — 카운트는 TMB 와 강하게 얽혀 있다(lift 9.96배).
비율은 "변이가 많고 적고"를 제거하고 **구성만** 남긴다. 둘이 다른 정보를 담는지 확인한다.

In [3]:
def fit_spec(df_fit):
    """전처리 규칙 학습. df_fit 에는 절대 test 를 넣지 않는다."""
    mask = df_fit[gene_cols].fillna("WT").values != "WT"
    return {"keep_idx": np.flatnonzero(mask.any(axis=0)).tolist(), "seed": SEED}


def build_features(df, counts, spec, blocks=("G", "B", "V", "R")):
    """blocks 에 지정된 블록만 이어붙인 희소 행렬과 피처 이름을 반환한다."""
    parts, names = [], []

    if "G" in blocks:
        keep = np.array(spec["keep_idx"])
        m = (df[gene_cols].fillna("WT").values != "WT")[:, keep]
        parts.append(sparse.csr_matrix(m.astype(np.float32)))
        names += [f"A_gene__{gene_cols[i]}" for i in keep]

    if "B" in blocks:
        b = np.log1p(counts[["n_mut_genes", "n_events", "n_multi_genes"]].values)
        parts.append(sparse.csr_matrix(b.astype(np.float32)))
        names += ["A_burden__log_genes", "A_burden__log_events", "A_burden__log_multi"]

    if "V" in blocks:
        v = np.log1p(counts[KINDS].values)
        parts.append(sparse.csr_matrix(v.astype(np.float32)))
        names += [f"A_vcount__{k}" for k in KINDS]

    if "R" in blocks:
        tot = counts[KINDS].values.sum(axis=1, keepdims=True)
        r = np.divide(counts[KINDS].values, tot, out=np.zeros_like(counts[KINDS].values),
                      where=tot > 0)
        parts.append(sparse.csr_matrix(r.astype(np.float32)))
        names += [f"A_vratio__{k}" for k in KINDS]

    return sparse.hstack(parts, format="csr"), names


spec_full = fit_spec(train)
X_demo, names_demo = build_features(train, cnt_train, spec_full)
print(f"전체 블록 사용 시 {X_demo.shape}  밀도 {X_demo.nnz / np.prod(X_demo.shape) * 100:.2f}%")
print("피처 이름 예시:", names_demo[:2], "...", names_demo[-3:])

전체 블록 사용 시 (6201, 4245)  밀도 1.02%
피처 이름 예시: ['A_gene__A2M', 'A_gene__AAAS'] ... ['A_vratio__frameshift', 'A_vratio__indel', 'A_vratio__other']


## Section 4 · Leakage 자가검증

In [4]:
checks = []

# ① 부분집합 불변성 — 앞 100행만 넣어도 전체의 앞 100행과 같아야 한다
part_cnt = parse_sample_counts(test.iloc[:100])
part_X, _ = build_features(test.iloc[:100], part_cnt, spec_full)
full_X, _ = build_features(test, cnt_test, spec_full)
checks.append(("부분집합 불변성 (100행)",
               np.array_equal(part_X.toarray(), full_X[:100].toarray())))

# ② 단일 행 추론
one_cnt = parse_sample_counts(test.iloc[[7]])
one_X, _ = build_features(test.iloc[[7]], one_cnt, spec_full)
checks.append(("단일 행 독립성", np.array_equal(one_X.toarray(), full_X[[7]].toarray())))

# ③ spec 은 train 만으로 유도
checks.append(("spec 재현성", fit_spec(train)["keep_idx"] == spec_full["keep_idx"]))

# ④ 유한값
checks.append(("NaN·inf 없음", bool(np.isfinite(full_X.data).all())))

# ⑤ test 결측 대응 — fillna 없이는 값이 달라져야 정상 (처리하고 있다는 증거)
checks.append(("test 결측 237개를 fillna 로 처리", int(test.isna().sum().sum()) > 0))

for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
assert all(ok for _, ok in checks), "Leakage 자가검증 실패"

  [PASS] 부분집합 불변성 (100행)
  [PASS] 단일 행 독립성
  [PASS] spec 재현성
  [PASS] NaN·inf 없음
  [PASS] test 결측 237개를 fillna 로 처리


## Section 5 · Ablation

**fold 안에서 spec 을 다시 fit** 한다 (협업 규정 2). 모델은 `LogisticRegression(balanced)` 로 고정해
**피처 효과만** 분리한다.

In [5]:
MODEL_PARAMS = dict(max_iter=1000, class_weight="balanced")


def run_cv(blocks, label, model_fn=None):
    if model_fn is None:
        model_fn = lambda: LogisticRegression(**MODEL_PARAMS)
    t = time.time()
    oof = np.empty(len(y_all), dtype=object)
    for i_tr, i_va in cv.split(train, y_all):
        spec = fit_spec(train.iloc[i_tr])                  # ← fold 안에서 fit
        Xa, _ = build_features(train.iloc[i_tr], cnt_train.iloc[i_tr], spec, blocks)
        Xb, _ = build_features(train.iloc[i_va], cnt_train.iloc[i_va], spec, blocks)
        clf = model_fn().fit(Xa, y_all[i_tr])
        oof[i_va] = clf.predict(Xb)
    oof = np.array(list(oof))
    f1 = f1_score(y_all, oof, average="macro")
    acc = accuracy_score(y_all, oof)
    dim = Xa.shape[1]
    print(f"{label:34} dim {dim:5d}   Macro F1 {f1:.5f}   Acc {acc:.5f}   ({time.time() - t:.0f}s)")
    return {"label": label, "blocks": "".join(blocks), "dim": dim,
            "f1_macro": round(float(f1), 5), "accuracy": round(float(acc), 5), "oof": oof}


ABLATION = [
    (("G",),                    "G        유전자만"),
    (("G", "B"),                "G+B      + 변이 부담"),
    (("G", "B", "V"),           "G+B+V    + 유형 카운트"),
    (("G", "B", "V", "R"),      "G+B+V+R  + 유형 비율"),
    (("B", "V", "R"),           "B+V+R    유전자 없이 집계만"),
]
results = [run_cv(b, l) for b, l in ABLATION]

G        유전자만                      dim  4226   Macro F1 0.34469   Acc 0.34317   (17s)
G+B      + 변이 부담                   dim  4229   Macro F1 0.37429   Acc 0.36365   (18s)
G+B+V    + 유형 카운트                  dim  4235   Macro F1 0.38218   Acc 0.37558   (20s)
G+B+V+R  + 유형 비율                   dim  4241   Macro F1 0.38389   Acc 0.37671   (21s)
B+V+R    유전자 없이 집계만                dim    15   Macro F1 0.15581   Acc 0.17336   (8s)


In [6]:
base = 0.36305      # member-d-logreg-001
tab = pd.DataFrame([{k: r[k] for k in ("label", "dim", "f1_macro", "accuracy")} for r in results])
tab["기준선 대비"] = (tab["f1_macro"] - base).round(5)
tab["판정"] = np.where(tab["f1_macro"].round(3) > round(base, 3), "향상",
              np.where(tab["f1_macro"].round(3) < round(base, 3), "하락", "동일"))
print(tab.to_string(index=False))
print()
print(f"※ 판정은 소수점 셋째 자리 반올림 기준 (협업 규정 2). 기준선 {base:.5f} → {base:.3f}")
tab.to_csv(ARTIFACTS / f"{NB}_ablation.csv", index=False, encoding="utf-8-sig")

              label  dim  f1_macro  accuracy   기준선 대비 판정
      G        유전자만 4226   0.34469   0.34317 -0.01836 하락
   G+B      + 변이 부담 4229   0.37429   0.36365  0.01124 향상
  G+B+V    + 유형 카운트 4235   0.38218   0.37558  0.01913 향상
   G+B+V+R  + 유형 비율 4241   0.38389   0.37671  0.02084 향상
B+V+R    유전자 없이 집계만   15   0.15581   0.17336 -0.20724 하락

※ 판정은 소수점 셋째 자리 반올림 기준 (협업 규정 2). 기준선 0.36305 → 0.363


## Section 6 · 어느 클래스가 좋아졌나

가설이 맞다면 **유형 구성이 극단적인 클래스**가 올라야 한다 —
LAML(frameshift 15.24%), DLBC·ACC·SKCM(silent 33\~41%), THYM(silent 17.68%).

In [7]:
classes = sorted(pd.unique(y_all))
best = max(results, key=lambda r: r["f1_macro"])
gb = next(r for r in results if r["blocks"] == "GB")

f1_gb = f1_score(y_all, gb["oof"], average=None, labels=classes)
f1_best = f1_score(y_all, best["oof"], average=None, labels=classes)

delta = pd.DataFrame({"G+B": f1_gb, best["blocks"]: f1_best}, index=classes)
delta["변화"] = (delta[best["blocks"]] - delta["G+B"]).round(4)
delta = delta.round(4).sort_values("변화", ascending=False)

print(f"최고 조합: {best['label'].strip()}  ({best['f1_macro']:.5f})")
print()
print("가장 오른 클래스 8개")
print(delta.head(8).to_string())
print()
print("가장 내린 클래스 5개")
print(delta.tail(5).to_string())

WATCH = ["LAML", "DLBC", "ACC", "SKCM", "THYM"]
print()
print("가설 대상 클래스 (유형 구성이 극단적인 곳)")
print(delta.loc[[c for c in WATCH if c in delta.index]].to_string())

delta.to_csv(ARTIFACTS / f"{NB}_class_f1_delta.csv", encoding="utf-8-sig")

최고 조합: G+B+V+R  + 유형 비율  (0.38389)

가장 오른 클래스 8개
         G+B    GBVR      변화
LUSC  0.2884  0.4037  0.1153
CESC  0.1221  0.1805  0.0584
PAAD  0.1942  0.2466  0.0524
STES  0.3476  0.3848  0.0372
SARC  0.1721  0.2057  0.0336
LUAD  0.2353  0.2655  0.0302
BRCA  0.4842  0.5104  0.0262
OV    0.3477  0.3633  0.0157

가장 내린 클래스 5개
         G+B    GBVR      변화
THYM  0.2881  0.2763 -0.0118
KIRC  0.1326  0.1152 -0.0174
ACC   0.8421  0.8182 -0.0239
DLBC  0.4643  0.4231 -0.0412
BLCA  0.3399  0.2857 -0.0542

가설 대상 클래스 (유형 구성이 극단적인 곳)
         G+B    GBVR      변화
LAML  0.5423  0.5382 -0.0040
DLBC  0.4643  0.4231 -0.0412
ACC   0.8421  0.8182 -0.0239
SKCM  0.7181  0.7302  0.0122
THYM  0.2881  0.2763 -0.0118


## Section 7 · 모델 비교

최고 피처셋을 고정하고 모델만 바꾼다. 전부 희소 행렬에서 빠른 선형 계열이다.
(GBDT 는 4,200차원 × 26클래스라 느려서 별도 실험으로 뺀다.)

In [8]:
best_blocks = tuple(best["blocks"])
MODELS = [
    ("LogisticRegression(balanced)", lambda: LogisticRegression(**MODEL_PARAMS)),
    ("LinearSVC(balanced)",          lambda: LinearSVC(class_weight="balanced", max_iter=3000)),
    ("SGD(modified_huber, balanced)", lambda: SGDClassifier(
        loss="modified_huber", class_weight="balanced", max_iter=3000,
        random_state=SEED, n_jobs=-1)),
]
model_results = [run_cv(best_blocks, name, fn) for name, fn in MODELS]

LogisticRegression(balanced)       dim  4241   Macro F1 0.38389   Acc 0.37671   (20s)
LinearSVC(balanced)                dim  4241   Macro F1 0.30390   Acc 0.30560   (27s)
SGD(modified_huber, balanced)      dim  4241   Macro F1 0.31715   Acc 0.32092   (13s)


## Section 8 · 최종 학습 · submission · metrics.json

In [9]:
champion = max(results + model_results, key=lambda r: r["f1_macro"])
model_name = champion["label"].strip()
model_fn = dict(MODELS).get(model_name, lambda: LogisticRegression(**MODEL_PARAMS))
final_blocks = tuple(champion["blocks"])

EXPERIMENT_ID = "member-d-logreg-002"        # 최고 모델이 바뀌면 토큰도 바꿀 것

t = time.time()
spec_final = fit_spec(train)                 # 최종 학습만 전체 train 으로 fit
Xtr, feat_names = build_features(train, cnt_train, spec_final, final_blocks)
Xte, _ = build_features(test, cnt_test, spec_final, final_blocks)
final_model = model_fn().fit(Xtr, y_all)
pred = final_model.predict(Xte)
print(f"최종 학습+추론 {time.time() - t:.0f}s · 등장 클래스 {len(set(pred))}/26")

sub = submission.copy()
sub[TARGET] = pred
sub_checks = [
    ("행 수 일치", len(sub) == len(submission)),
    ("컬럼 [ID, SUBCLASS]", list(sub.columns) == [ID, TARGET]),
    ("ID 순서 일치", bool((sub[ID].values == test[ID].values).all())),
    ("결측 없음", int(sub.isna().sum().sum()) == 0),
    ("train 클래스 안", set(sub[TARGET]).issubset(set(y_all))),
    ("한 클래스 쏠림 없음", sub[TARGET].value_counts(normalize=True).iloc[0] < 0.40),
]
for n_, ok in sub_checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {n_}")
assert all(ok for _, ok in sub_checks)

최종 학습+추론 5s · 등장 클래스 26/26
  [PASS] 행 수 일치
  [PASS] 컬럼 [ID, SUBCLASS]
  [PASS] ID 순서 일치
  [PASS] 결측 없음
  [PASS] train 클래스 안
  [PASS] 한 클래스 쏠림 없음


In [10]:
RESULT_DIR = ROOT / "experiments" / MEMBER / "results" / EXPERIMENT_ID
RESULT_DIR.mkdir(parents=True, exist_ok=True)
sub.to_csv(RESULT_DIR / "submission.csv", index=False, encoding="UTF-8-sig")

metrics = {
    "experiment": EXPERIMENT_ID,
    "owner": MEMBER,
    "track": "A",
    "model": model_name,
    "seed": SEED,
    "validation": "StratifiedKFold-5",
    "accuracy": champion["accuracy"],
    "f1_macro": champion["f1_macro"],
    "n_features": int(Xtr.shape[1]),
    "feature_blocks": champion["blocks"],
    "hyperparameters": MODEL_PARAMS,
    "preprocessing": [
        "fillna('WT')",
        "cell-level duplicate token removal",
        "variant classification: > | fs | del/ins | ^([A-Z*]+)(digits)(.*)$",
        "gene binary (train-fold non-constant only)",
        "log1p burden (genes / events / multi-event genes)",
        "log1p variant-type counts",
        "variant-type ratios",
        "scipy.sparse CSR",
    ],
    "baseline": {"experiment": "member-d-logreg-001", "f1_macro": base},
    "environment": {
        "python": platform.python_version(), "platform": platform.platform(),
        "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__,
    },
    "description": "트랙 A — 변이 유형 집계 피처. 유전자 이진화가 버리는 유형 정보를 복원.",
}
(RESULT_DIR / "metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

spec_out = {"version": "A_v1_variant_type", "n_features": int(Xtr.shape[1]),
            "blocks": champion["blocks"], "feature_names_head": feat_names[:5],
            "feature_names_tail": feat_names[-12:], "keep_idx": spec_final["keep_idx"]}
(ARTIFACTS / f"{NB}_features_A_spec.json").write_text(
    json.dumps(spec_out, ensure_ascii=False, indent=2), encoding="utf-8")

print("submission :", RESULT_DIR / "submission.csv")
print("metrics    :", RESULT_DIR / "metrics.json")
print()
print("=" * 70)
print(f"실험 ID    : {EXPERIMENT_ID}")
print(f"트랙       : A · 변이 표기 활용")
print(f"모델       : {model_name}")
print(f"피처       : {champion['blocks']} · {Xtr.shape[1]}차원")
print(f"Macro F1   : {champion['f1_macro']:.5f}   (기준선 {base:.5f})")
print(f"Accuracy   : {champion['accuracy']:.5f}")
print(f"판정       : {'향상' if round(champion['f1_macro'],3) > round(base,3) else ('하락' if round(champion['f1_macro'],3) < round(base,3) else '동일')}  (셋째 자리 기준)")
print("=" * 70)

submission : /Volumes/DataSSD/AI_health_care_5/25_aiCancerClassification/experiments/member_d/results/member-d-logreg-002/submission.csv
metrics    : /Volumes/DataSSD/AI_health_care_5/25_aiCancerClassification/experiments/member_d/results/member-d-logreg-002/metrics.json

실험 ID    : member-d-logreg-002
트랙       : A · 변이 표기 활용
모델       : G+B+V+R  + 유형 비율
피처       : GBVR · 4245차원
Macro F1   : 0.38389   (기준선 0.36305)
Accuracy   : 0.37671
판정       : 향상  (셋째 자리 기준)


### ✅ 실행 후 확인할 것

1. **Section 5 ablation 표** — `G+B` 가 기준선 0.363 을 재현하는가. 안 되면 파서 규칙 ①(중복 제거) 때문이므로 그 자체가 결과다
2. **`V` 를 더했을 때 오르는가** — 이 노트북의 핵심 질문
3. **`R`(비율)이 `V`(카운트) 위에 더 얹어지는가** — 둘이 다른 정보인지
4. **`B+V+R` 만으로 몇 점인가** — 15차원으로 4,230차원의 몇 %를 설명하는지
5. **Section 6에서 LAML·DLBC·ACC·SKCM 이 올랐는가** — 가설의 직접 검증

> 결과를 알려주시면 해석하고, 노션 【📋 실험·제출 기록】에 올리겠습니다.
> `EXPERIMENT_ID` 는 `member-d-logreg-002` 로 고정해뒀는데, 최고 모델이 LinearSVC 나 SGD 로 나오면
> `member-d-svm-002` / `member-d-sgd-002` 로 바꿔야 합니다 (README 네이밍 규칙).
